# MB01 — Frozen reference dependency verification

**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

This notebook verifies copied reference artifacts. Original MB01 training is archived in the backup, not rerun.

In [1]:
from pathlib import Path
import os, json, hashlib, sys, importlib.util
ROOT=Path(os.environ['SOCCER_MI_ROOT']) if 'SOCCER_MI_ROOT' in os.environ else next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate/'model_training_evaluation').is_dir() and (candidate/'mi_analysis').is_dir()), None)
assert ROOT is not None, 'Run from within the project or set SOCCER_MI_ROOT.'
MI_DIR=Path(os.environ.get('MI_ANALYSIS_DIR',ROOT/'mi_analysis')).resolve()
sys.path.insert(0,str(ROOT/'model_training_evaluation'))
spec=importlib.util.spec_from_file_location('mi_src',MI_DIR/'src/__init__.py',submodule_search_locations=[str(MI_DIR/'src')])
mi_src=importlib.util.module_from_spec(spec);sys.modules['mi_src']=mi_src;spec.loader.exec_module(mi_src)
from mi_src import fingerprint
manifest=MI_DIR/'manifests/sha_manifest.json'
backup=ROOT/'mi_analysis_backup_2026-10-04'
files={str(p.relative_to(MI_DIR)):p for p in sorted((MI_DIR/'artifacts/conflict').rglob('*')) if p.is_file()}
for name,p in files.items():
    source=backup/name
    assert fingerprint.file_sha256(p)==fingerprint.file_sha256(source),name
fingerprint.record_files(manifest,files)
fingerprint.record_files(manifest,{'events_prepared.parquet':ROOT/'model_training_evaluation/artifacts/events_prepared.parquet'})
import pandas as pd
conflict_set=pd.read_parquet(MI_DIR/'artifacts/conflict/conflict_set.parquet')
fingerprint.record_value(manifest,'conflict_set_content',fingerprint.rowset_sha256(conflict_set['event_id']))
fingerprint.write_snapshot(MI_DIR/'manifests/track_a_snapshot.json',[ROOT/'model_training_evaluation/src',ROOT/'model_training_evaluation/config',ROOT/'model_training_evaluation/benchmarks/track_a_4layer/config'])
fingerprint.write_snapshot(MI_DIR/'manifests/run_input_snapshot.json',[MI_DIR/'src',MI_DIR/'tests',MI_DIR/'config'])
fingerprint.record_files(manifest,{str(p.relative_to(MI_DIR)):p for p in (MI_DIR/'src').glob('*.py')})
print('Verified',len(files),'frozen reference dependencies; conflict rows',len(conflict_set))
print(json.loads((MI_DIR/'config/analysis_plan.json').read_text())['status'])


Verified 13 frozen reference dependencies; conflict rows 21792
NEW EXPLORATORY BENCHMARK ANALYSIS; not replication of historical SEL-4L wd0.01; test split previously accessed
